# A Predictive Model for US Airline Flight Delays - EDA

### <span style="color:chocolate"> Project Description </span>

Flight delay prediction is an important research field, as flight delays are the primary concern of aviation
stakeholders. Our project aims to identify the predictors of flight delays before they happen, improve
travelers experience, and help airlines shift from reactive damage control to proactive delay management.

---
### <span style="color:chocolate">Import libraries</span>

In [43]:
import numpy as np
import pandas as pd
import glob
import os
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split

---
### <span style="color:chocolate">Data Ingestion and Data Preprocessing</span>

Reading the data from 2022 to 2026 to the data frame downloaded from the following data sources:
- Flight data: Bureau of transportation stats https://www.transtats.bts.gov/ontime/
- Weather data (New with 2026 data): https://www.ncei.noaa.gov/access/search/data-search/global-historical-climatology-network-hourly
- Aircraft data: https://www.faa.gov/licenses_certificates/aircraft_certification/aircraft_registry/releasable_aircraft_download?utm_source=chatgpt.com

In [52]:
# Read in all weather data in the psv files to the data frame and indicate 'I' as the delimiter.
weather_files_path = "Data/*.psv"
all_weather_files = glob.glob(weather_files_path)
weather = pd.concat((pd.read_csv(f, sep="|") for f in all_weather_files), ignore_index=True)

# Read in all flight data to the data frame. The first 7 lines that contains the data information are skipped.
flights_files_path = "Data/DFW_Arrivals_*.csv"
all_flights_files = glob.glob(flights_files_path)
flights = pd.concat((pd.read_csv(f, skiprows=7) for f in all_flights_files), ignore_index=True)

# Read in all aircraft data to the data frame.
aircraft = pd.read_csv('Data/ReleasableAircraft/MASTER.txt', sep=',')

/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_87033/3261188870.py:4: DtypeWarning: Columns (43,55,77,80,82,97,103,128,130,133,217,223,229,241,247,253) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.concat((pd.read_csv(f, sep="|") for f in all_weather_files), ignore_index=True)
/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_87033/3261188870.py:4: DtypeWarning: Columns (13,19,43,55,77,80,82,91,97,103,133,217,223,229,241,247,253) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.concat((pd.read_csv(f, sep="|") for f in all_weather_files), ignore_index=True)
/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_87033/3261188870.py:4: DtypeWarning: Columns (61,67,71,74,76,77,80,82,89,92,94,95,98,100,101,104,106,113,116,118,119,122,124,240,246,252,320,322) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.concat((pd.read_csv(f, sep="|") for f i

In [53]:
# Convert the "DATE" column in weather data frame to datetime object.
weather["Weather_Datetime"] = pd.to_datetime(weather["DATE"])
# Combine the "Date (MM/DD/YYYY)" and "Scheduled Arrival Time" columns into a single column for joining the weather data frame.
flights["Full_Time_Str"] = flights["Date (MM/DD/YYYY)"] + ' ' + flights["Scheduled Arrival Time"]
# Convert the "Full_Time_Str" column created above to datatime object and round it to the nearest hour to match the time in weather data frame.
flights["Flight_Hourly"] = pd.to_datetime(flights["Full_Time_Str"]).dt.round("h")
# Remove the alphabet 'N' from the "Tail Number" column in flights data frame to match the "N_Num" in aircraft data frame.
flights["Join_N_Num"] = flights["Tail Number"].str.lstrip("N")

# Join the flights and weather data frames together by using left join and the data time as key.
merged_df = pd.merge(
    flights,
    weather,
    left_on='Flight_Hourly',
    right_on='Weather_Datetime',
    how='left'
)

# Join the flights and aircraft data frames together by using left join and the data time as key.
final_df = pd.merge(
    merged_df,
    aircraft,
    left_on='Join_N_Num',
    right_on='N-NUMBER',
    how='left'
)

# Delete the columns that created above to join the data frames.
final_df = final_df.drop(columns=["Full_Time_Str", "Flight_Hourly", "Weather_Datetime", "Join_N_Num"])

/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_87033/2627168220.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather["Weather_Datetime"] = pd.to_datetime(weather["DATE"])


In [54]:
# Replace the space in the column names with underline.
final_df.columns = final_df.columns.str.strip().str.lower().str.replace(" " , "_")

,carrier_code,date_(mm/dd/yyyy),flight_number,tail_number,origin_airport,scheduled_arrival_time,actual_arrival_time,scheduled_elapsed_time_(minutes),actual_elapsed_time_(minutes),arrival_delay_(minutes),...,other_names(2),other_names(3),other_names(4),other_names(5),expiration_date,unique_id,kit_mfr,kit_model,mode_s_code_hex,unnamed:_34
0,DL,01/01/2022,531.0,N310DU,LGA,17:19,17:20,259.0,264.0,1.0,...,...,...,...,...,20281231,1355999.0,,,A3472F,NaN
1,DL,01/01/2022,561.0,N307DU,LGA,11:57,11:35,262.0,246.0,-22.0,...,...,...,...,...,20280131,1334505.0,,,A339B1,NaN
2,DL,01/01/2022,1036.0,N106DU,JFK,19:33,19:55,253.0,266.0,22.0,...,...,...,...,...,20270930,1291625.0,,,A01B5C,NaN
3,DL,01/01/2022,1041.0,N344NW,LAX,13:50,13:26,180.0,160.0,-24.0,...,...,...,...,...,20290630,924212.0,,,A3CD6B,NaN
4,DL,01/01/2022,1094.0,N326US,LAX,21:58,00:00,178.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
final_df.dropna(subset=["date_(mm/dd/yyyy)", "origin_airport"], inplace=True)

# Fills missing values in every numeric column with that column's average
df_filled = final_df.fillna(final_df.mean(numeric_only=True))

# Target specific columns
target_cols = ["temperature", "wind_speed", "relative_humidity"]

# Fill missing values in only those columns using their respective means
final_df[target_cols] = final_df[target_cols].fillna(final_df[target_cols].mean())

# Fills missing values using the last valid value from the row above
df_filled = final_df.ffill()

In [60]:
# Define the target list using the exact cleaned snake_case names

features_columns = [
    "date_(mm/dd/yyyy)",
    "origin_airport",
    "scheduled_arrival_time",
    "actual_arrival_time",
    "scheduled_elapsed_time_(minutes)",
    "actual_elapsed_time_(minutes)",
    "wheels-on_time",
    "taxi-in_time_(minutes)",
    "temperature",
    "sea_level_pressure",
    "wind_speed",
    "relative_humidity",
    "snow_depth",
    "sky_condition",
    "year_mfr",
    "type_engine",
]

In [61]:
df_cleaned = final_df.dropna(subset=features_columns)

In [26]:
X = final_df[features_columns].copy()
Y = final_df["arrival_delay_(minutes)"].copy()

In [ ]:
# Create training, validation, and test datasets using a 60/20/20 split
# Partition the (X, Y) data into training, validation, and test sets using a splitting rule of [60%, 20%, 20%], with a random state set to 1234.
# First, split the data into test_temp and train using a splitting rule of [60%, 40%] on the original data.
# After that, split the data into test and validation using a splitting rule of [50%] on the test_temp data.
X_train, X_test_temp, Y_train_processed, Y_test_temp = train_test_split(X, Y, test_size=0.4, random_state=1234)
X_val, X_test, Y_val_processed, Y_test_processed = train_test_split(X_test_temp, Y_test_temp, test_size=0.5, random_state=1234)

---
#### <span style="color:chocolate">Exploratory Data Analysis (EDA)</span>

In [ ]:
# Check if there is any null values in the dataframe
flights.isnull().any()

In [ ]:
# Calculate the total number of missing value in each column.
tail_num = flights['tail_number'].isna().sum()
scheduled_dep = flights['scheduled_departure_time'].isna().sum()
actual_dep = flights['actual_departure_time'].isna().sum()

print('Missing tail number value:', tail_num)
print('Missing scheduled departure time:', scheduled_dep)
print('Missing actual departure time:', actual_dep)

In [ ]:
flights.head()

In [ ]:
# Set up the matplotlib figure
plt.figure(figsize=(10, 5))

# Plot the flight count histogram
sns.countplot(data=flights, x='carrier_code', hue='carrier_code', palette='viridis', color='gold')
plt.title('AirLine Flight Count', fontsize=16, pad=15)
plt.xlabel('Air line', fontsize=10)
plt.ylabel('Flight Count', fontsize=10)

# Adjust padding parameters of the subplot and display the plots.
plt.tight_layout()
plt.show()

In [ ]:
# Set up the matplotlib figure
plt.figure(figsize=(12, 5))  # Slightly widened to give bars more breathing room

# Create a histogram for destination count
sns.countplot(data=flights, x='destination_airport', hue='destination_airport', palette='viridis')
plt.title('Destination Flight Count', fontsize=16, pad=15)
# Rotate the x label to 90 degrees with center alignment
plt.xticks(rotation=90, ha='center', fontsize=8)
# Label x and y axes.
plt.xlabel('Destination Airport', fontsize=10)
plt.ylabel('Flight Count', fontsize=10)
# Display the histogram
plt.tight_layout()
plt.show()

In [ ]:
# Extract Year and Month
flights['Year'] = flights['date_(mm/dd/yyyy)'].dt.year
flights['Month'] = flights['date_(mm/dd/yyyy)'].dt.month
flights['Month_Name'] = flights['date_(mm/dd/yyyy)'].dt.strftime('%b')

# Group the data to count flights for each month in each year
monthly_counts = (
    flights.groupby(['Year', 'Month', 'Month_Name'])
    .size()
    .reset_index(name='Flight_Count')
)

# Create the chart
g = sns.relplot(
    data=monthly_counts,
    x='Month_Name',
    y='Flight_Count',
    col='Year',
    col_wrap=2,
    kind='line',
    marker='o',
    linewidth=2.5,
    color='teal',
    height=4,
    aspect=1.5,
    facet_kws=dict(sharex=False),
)

# Label and configure the chart
g.set_titles('{col_name} Monthly Flight Trends')
g.set_axis_labels('Month', 'Flights Count')
for ax in g.axes.flat:
    ax.tick_params(axis='x', labelbottom=True, labelrotation=45)
    ax.grid(True, linestyle='--', alpha=0.6)

# Plot the chart
plt.tight_layout()
plt.show()

---
#### <span style="color:chocolate">Data splits</span>

Partition the data into training, validation, and test sets using a splitting rule of [60%, 20%, 20%] in chronological order. Name the resulting dataframes as follows:

- train
- val
- test

In [ ]:
# Sort the data in the flight dataframe by date.
flights = flights.sort_values('date_(mm/dd/yyyy)').reset_index(drop=True)

In [ ]:
# Calculate the number of samples based on the split percentage then save the indexes number.
total_rows = len(flights)
train_end_idx = int(total_rows * 0.60)
test_end_idx = train_end_idx + int(total_rows * 0.20)

# Split the data
train = flights.iloc[:train_end_idx].copy()
test = flights.iloc[train_end_idx:test_end_idx].copy()
val = flights.iloc[test_end_idx:].copy()

In [ ]:
# Verify the date range is split correctly
print(f"Train: {train['date_(mm/dd/yyyy)'].min()} to {train['date_(mm/dd/yyyy)'].max()}")
print(f"Test: {test['date_(mm/dd/yyyy)'].min()} to {test['date_(mm/dd/yyyy)'].max()}")
print(f"Val: {val['date_(mm/dd/yyyy)'].min()} to {val['date_(mm/dd/yyyy)'].max()}")

---
#### <span style="color:chocolate">Data Exploratory Data Analysis (EDA)</span>

In [ ]:
# Pick out the numerical data column for analysis
timedelta_cols = [
    'scheduled_elapsed_time_(minutes)', 'actual_elapsed_time_(minutes)',
    'departure_delay_(minutes)', 'taxi-out_time_(minutes)',
    'delay_carrier_(minutes)', 'delay_weather_(minutes)',
    'delay_national_aviation_system_(minutes)', 'delay_security_(minutes)',
    'delay_late_aircraft_arrival_(minutes)'
]
# Create the correlation matrix
for col in timedelta_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')

train[timedelta_cols] = train[timedelta_cols].fillna(0)

correlation_matrix = train[timedelta_cols].corr()
# Create a mask to filter out the repeated output.
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

# Set up the matplotlib
plt.figure(figsize=(12, 10))

# Configure the correlation matrix
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    cmap='mako',
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)

plt.show()

In [ ]:
# Get the flight counts based on tail number.
flight_counts = train['tail_number'].value_counts()

# Calculate the average delay by tail number then only select the top 10.
top_delayed_tail_num = (
    train.groupby('tail_number')['delay_late_aircraft_arrival_(minutes)']
    .mean()
    .nlargest(10)
)

# Plot the chart
plt.figure(figsize=(10, 6))

# Add labels and Configure the chart
top_delayed_tail_num.plot(kind='bar', color='salmon', edgecolor='black')
plt.title('Top 10 Airplane Tail Numbers with the Worst Average Delays', fontsize=14, pad=15)
plt.ylabel('Average Departure Delay (Minutes)', fontsize=12)
plt.xlabel('Tail Number', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Calculate average delay for each destination and select the top 30 to include in the chart
top_delayed_airports = (
    train.groupby('destination_airport')['delay_late_aircraft_arrival_(minutes)']
    .mean()
    .nlargest(30)
)

# Plot the chart
plt.figure(figsize=(12, 8))

# Add labels and Configure the chart
top_delayed_airports.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Top Destination Airports with the Highest Average Departure Delays', fontsize=14, pad=15)
plt.xlabel('Average Departure Delay (Minutes)', fontsize=12)
plt.ylabel('Destination Airport', fontsize=12)
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Display the plot
plt.tight_layout()
plt.show()